In [ ]:
!pip install gymnasium
!pip install moviepy

import gymnasium as gym
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import random
from collections import deque
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter
from IPython.display import HTML
from base64 import b64encode

# Инициализация среды Acrobot
env = gym.make("Acrobot-v1", render_mode="rgb_array")

episodes = 500
epsilon = 1.0  # Начальное значение epsilon
epsilon_min = 0.01  # Минимальное значение epsilon
epsilon_decay = 0.995  # Скорость уменьшения epsilon
gamma = 0.99  # Коэффициент дисконтирования для будущих наград
batch_size = 32
learning_rate = 0.001
memory = deque(maxlen=10000)  # Очередь для хранения опыта
update_frequency = 4  # Частота обновления модели

# Создание модели DQN
state_size = env.observation_space.shape[0]  # Размер состояния
action_size = env.action_space.n  # Количество возможных действий

# Архитектура нейронной сети
model = tf.keras.Sequential([
    layers.Dense(128, input_dim=state_size, activation='relu'),  # Полносвязный слой с 128 нейронами
    layers.Dense(128, activation='relu'),  # Второй полносвязный слой с 128 нейронами
    layers.Dense(128, activation='relu'),  # Третий полносвязный слой с 128 нейронами
    layers.Dense(action_size, activation='linear')  # Выходной слой с количеством нейронов, равным количеству действий
])


model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

# Функция для выбора действия
def choose_action(state, epsilon):
    if np.random.rand() <= epsilon:
        return env.action_space.sample()  # Случайное действие
    q_values = model.predict(state, verbose=0)  # Предсказание Q-значений для текущего состояния
    return np.argmax(q_values[0])  # Жадное действие

# Функция для запоминания опыта
def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

# Функция для обучения модели
def replay(batch_size):
    if len(memory) < batch_size:
        return  # Если памяти недостаточно для обучения, пропустить
    minibatch = random.sample(memory, batch_size)  # Случайный выбор мини-батча из памяти
    states = np.array([i[0] for i in minibatch])  # Состояния
    actions = np.array([i[1] for i in minibatch])  # Действия
    rewards = np.array([i[2] for i in minibatch])  # Награды
    next_states = np.array([i[3] for i in minibatch])  # Следующие состояния
    dones = np.array([i[4] for i in minibatch])  # Флаги завершения эпизода

    states = np.squeeze(states)  # Удаление лишних измерений
    next_states = np.squeeze(next_states)  # Удаление лишних измерений

    # Предсказание Q-значений для следующего состояния
    targets = rewards + gamma * (np.amax(model.predict_on_batch(next_states), axis=1)) * (1 - dones)
    targets_full = model.predict_on_batch(states)

    # Обновление Q-значений для выбранных действий
    ind = np.array([i for i in range(batch_size)])
    targets_full[[ind], [actions]] = targets

    # Обучение модели на мини-батче
    model.fit(states, targets_full, epochs=1, verbose=0)

# Функция для записи видео
def record_video(env, model, video_filename):
    state, _ = env.reset()  # Сброс среды
    frames = []  # Список для хранения кадров
    done = False

    while not done:
        action = np.argmax(model.predict(np.reshape(state, [1, state_size]), verbose=0)[0])  # Выбор действия
        state, reward, terminated, truncated, _ = env.step(action)  # Шаг в среде
        done = terminated or truncated  # Проверка завершения эпизода
        frame = env.render()
        frames.append(frame)  # Сохранение кадра

    # Сохраняем видео
    if len(frames) > 0:
        height, width, _ = frames[0].shape  # Размеры кадра
        writer = FFMPEG_VideoWriter(video_filename, (width, height), fps=30)  # Создание видео-файла
        for frame in frames:
            writer.write_frame(frame)  # Запись кадров в видео
        writer.close()  # Закрытие видео-файла
        print(f"Video saved to {video_filename}")

# Обучение агента
for episode in range(episodes):
    state, _ = env.reset()
    state = np.reshape(state, [1, state_size])
    done = False
    total_reward = 0
    steps = 0

    while not done:
        action = choose_action(state, epsilon)  # Выбор действия
        next_state, reward, terminated, truncated, _ = env.step(action)  # Шаг в среде
        done = terminated or truncated  # Проверка завершения эпизода
        next_state = np.reshape(next_state, [1, state_size])  # Преобразование следующего состояния
        remember(state, action, reward, next_state, done)  # Сохранение опыта
        state = next_state  # Обновление текущего состояния
        total_reward += reward  # Обновление общей награды
        steps += 1  # Увеличение счетчика шагов

        # Уменьшение epsilon
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        # Обновление модели
        if steps % update_frequency == 0:
            replay(batch_size)

    print(f"Episode: {episode + 1}, Total Reward: {total_reward}, Steps: {steps}, Epsilon: {epsilon:.4f}")

    if (episode + 1) % 50 == 0:
        record_video(env, model, f"acrobot_episode_{episode + 1}.mp4")

record_video(env, model, "acrobot_final.mp4")

video_path = "acrobot_final.mp4"
with open(video_path, "rb") as video_file:
    video_data = video_file.read()
video_base64 = b64encode(video_data).decode("utf-8")

video_html = f"""
<video width="640" height="480" controls>
    <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
</video>
"""

display(HTML(video_html))

env.close()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Episode: 1, Total Reward: -387.0, Steps: 388, Epsilon: 0.1430
Episode: 2, Total Reward: -236.0, Steps: 237, Epsilon: 0.0436
Episode: 3, Total Reward: -226.0, Steps: 227, Epsilon: 0.0140
Episode: 4, Total Reward: -336.0, Steps: 337, Epsilon: 0.0100
Episode: 5, Total Reward: -248.0, Steps: 249, Epsilon: 0.0100
Episode: 6, Total Reward: -426.0, Steps: 427, Epsilon: 0.0100
Episode: 7, Total Reward: -500.0, Steps: 500, Epsilon: 0.0100
Episode: 8, Total Reward: -500.0, Steps: 500, Epsilon: 0.0100
Episode: 9, Total Reward: -209.0, Steps: 210, Epsilon: 0.0100
Episode: 10, Total Reward: -325.0, Steps: 326, Epsilon: 0.0100
Episode: 11, Total Reward: -175.0, Steps: 176, Epsilon: 0.0100
Episode: 12, Total Reward: -132.0, Steps: 133, Epsilon: 0.0100
Episode: 13, Total Reward: -159.0, Steps: 160, Epsilon: 0.0100
Episode: 14, Total Reward: -208.0, Steps: 209, Epsilon: 0.0100
Episode: 15, Total Reward: -227.0, Steps: 228, Epsilon: 0.0100
Episode: 16, Total Reward: -122.0, Steps: 123, Epsilon: 0.0100
E

KeyboardInterrupt: 

In [ ]:
record_video(env, model, "acrobot_final.mp4")

env.close()

Video saved to acrobot_final.mp4


In [ ]:
from IPython.display import HTML
from base64 import b64encode

video_path = "acrobot_final.mp4"

with open(video_path, "rb") as video_file:
    video_data = video_file.read()
video_base64 = b64encode(video_data).decode("utf-8")

video_html = f"""
<video width="640" height="480" controls>
    <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
</video>
"""

display(HTML(video_html))